# Project Pipeline

One growing integration checkpoint. Loads the project dataset **once**, then runs each
stage's `src/` helper against it, top to bottom. The full analysis and write-up for each
stage lives in the matching homework notebook.

| stage | helper | write-up |
|---|---|---|
| 06 cleaning | `src/cleaning.py` | `homework/homework06/` |
| 07 outliers | `src/outliers.py` | `notebooks/sensitivity_outliers.ipynb` · `docs/outliers.md` |
| 08 EDA | `src/eda.py` | `notebooks/eda.ipynb` |
| 09 features | `src/features.py` | `homework/homework09/` |

In [11]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# notebook lives in notebooks/; src/ and data/ are one level up
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

from src import cleaning, outliers, features
from src.eda import eda_summary

pd.set_option("display.max_columns", 100)

## Load data

The project dataset: `data/raw/card_tiers.csv` (one row per hobby-box parallel / insert /
autograph tier) joined to a few box-level fields from `data/raw/box_products.csv`. Built by
`src/build_raw_dataset.py`; schema in `docs/data_dictionary.md`.

In [12]:
ct = pd.read_csv(project_root / "data" / "raw" / "card_tiers.csv")
bp = pd.read_csv(project_root / "data" / "raw" / "box_products.csv")
df = ct.merge(
    bp[["product_id", "product_line", "packs_per_box", "retail_price_usd", "release_date"]],
    on="product_id", how="left",
)
print(df.shape)
df.head()

(194, 17)


,product_id,box_format,tier_name,tier_group,print_run,is_numbered,is_autograph,is_ssp,odds_pack,odds_box,est_value_usd,value_basis,source_note,product_line,packs_per_box,retail_price_usd,release_date
0,2025-26-topps-chrome-bkb-hobby,Hobby,Base,base,NaN,False,False,False,NaN,NaN,0.5,rough_estimate_v0,NaN,2025-26 Topps Chrome Basketball,20,379.99,2025-12-18
1,2025-26-topps-chrome-bkb-hobby,Hobby,Base Refractor,parallel,NaN,False,False,False,3.0,0.15,3.0,rough_estimate_v0,NaN,2025-26 Topps Chrome Basketball,20,379.99,2025-12-18
2,2025-26-topps-chrome-bkb-hobby,Hobby,Prism Refractor,parallel,NaN,False,False,False,5.0,0.25,3.0,rough_estimate_v0,NaN,2025-26 Topps Chrome Basketball,20,379.99,2025-12-18
3,2025-26-topps-chrome-bkb-hobby,Hobby,Wave Refractor,parallel,NaN,False,False,False,14.0,0.70,3.0,rough_estimate_v0,NaN,2025-26 Topps Chrome Basketball,20,379.99,2025-12-18
4,2025-26-topps-chrome-bkb-hobby,Hobby,Negative Refractor,parallel,NaN,False,False,False,31.0,1.55,8.0,rough_estimate_v0,NaN,2025-26 Topps Chrome Basketball,20,379.99,2025-12-18


## Stage 06 — cleaning  →  `src/cleaning.py`

`fill_missing_median` / `drop_missing` / `normalize_data` are available, but this dataset
needs **no row-level imputation** — the NaNs are structural, not dirty:

- `print_run` (~51% NaN) = the card is **unnumbered**, not missing. `is_numbered` encodes it; filling would invent print runs.
- `odds_pack` (~7% NaN) = Topps published **card-level** odds only for that tier, no aggregate. Left as NaN, handled per stage.

In [13]:
missing = df.isna().sum()
print(missing[missing > 0])
# Decision: keep the structural NaNs (see markdown above) -- no fill_missing_median /
# drop_missing here. cleaning.normalize_data is a modelling-time step, deferred to Stage 10.

print_run       99
odds_pack       13
odds_box        13
source_note    174
dtype: int64


## Stage 07 — outliers  →  `notebooks/sensitivity_outliers.ipynb` · `docs/outliers.md`

`src/outliers.py`: flag the rare "chase" tiers on the rarity axis (`odds_pack`). **Flag,
never remove** — those tiers carry most of a box's EV (`docs/outliers.md`).

In [14]:
mask_iqr = outliers.detect_outliers_iqr(df["odds_pack"])
df = outliers.flag_outliers(df, "odds_pack", mask_iqr, flag_column="is_chase")
print(f"chase tiers flagged: {int(mask_iqr.sum())} of {len(df)} ({mask_iqr.mean():.1%})  "
      f"-- flagged, not removed")
df.loc[df["is_chase"], ["product_line", "tier_name", "odds_pack", "est_value_usd"]].head(10)

chase tiers flagged: 29 of 194 (14.9%)  -- flagged, not removed


,product_line,tier_name,odds_pack,est_value_usd
21,2025-26 Topps Chrome Basketball,Red Refractor,4353.0,550.0
22,2025-26 Topps Chrome Basketball,Red Wave Refractor,3420.0,550.0
23,2025-26 Topps Chrome Basketball,FrozenFractor,4353.0,550.0
24,2025-26 Topps Chrome Basketball,SuperFractor,21767.0,4000.0
58,2025-26 Topps Chrome Update Series Basketball,Black Refractor,3022.0,300.0
60,2025-26 Topps Chrome Update Series Basketball,Red Refractor,6054.0,550.0
61,2025-26 Topps Chrome Update Series Basketball,Red Wave Refractor,3119.0,550.0
62,2025-26 Topps Chrome Update Series Basketball,FrozenFractor,6054.0,550.0
63,2025-26 Topps Chrome Update Series Basketball,SuperFractor,30361.0,4000.0
66,2025-26 Topps Chrome Update Series Basketball,Denim Tears,6054.0,150.0


## Stage 08 — EDA  →  `notebooks/eda.ipynb`

`src/eda.py`'s `eda_summary()` — quick profile of the project dataset.

In [15]:
summary = eda_summary(df)
print("shape:", summary["shape"])
print("missing:", {k: v for k, v in summary["missing"].items() if v})
summary["numeric_profile"]

shape: (194, 18)
missing: {'print_run': 99, 'odds_pack': 13, 'odds_box': 13, 'source_note': 174}


,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
print_run,95.0,96.105263,106.471300,1.00,10.00,50.00,150.00,499.00,1.362879,1.640604
odds_pack,181.0,3519.198895,13156.920371,1.00,76.00,286.00,1209.00,138789.00,7.416351,65.578591
odds_box,181.0,189.061050,668.443559,0.05,3.95,16.65,71.38,6939.45,7.093851,60.980879
est_value_usd,194.0,287.324742,815.751158,0.50,16.00,100.00,150.00,5000.00,4.414497,18.452395
packs_per_box,194.0,18.206186,4.289856,8.00,20.00,20.00,20.00,20.00,-1.966065,1.865413
retail_price_usd,194.0,667.651495,336.901344,259.95,379.99,549.99,1099.95,1100.00,0.373870,-1.595295


## Stage 09 — features  →  `homework/homework09/`

`src/features.py`: `expected_hits_per_box` (the EV weight), `log_odds_pack` (linear rarity
scale), and a one-hot of `tier_group`. Feature definitions in the README.

In [16]:
feat = features.encode_tier_group(
    features.add_log_odds(
        features.add_expected_hits_per_box(df)
    )
)
feat.filter(regex="expected_hits_per_box|log_odds_pack|is_chase|^tg_").describe().T

,count,mean,std,min,25%,50%,75%,max
expected_hits_per_box,181.0,0.609062,2.280459,0.000144,0.014011,0.060060,0.253165,20.000000
log_odds_pack,181.0,2.479064,0.988136,0.000000,1.880814,2.456366,3.082426,5.142355
tg_auto,194.0,0.242268,0.429564,0.000000,0.000000,0.000000,0.000000,1.000000
tg_base,194.0,0.041237,0.199353,0.000000,0.000000,0.000000,0.000000,1.000000
tg_insert,194.0,0.144330,0.352333,0.000000,0.000000,0.000000,0.000000,1.000000
tg_parallel,194.0,0.556701,0.498060,0.000000,0.000000,1.000000,1.000000,1.000000
tg_variation,194.0,0.015464,0.123708,0.000000,0.000000,0.000000,0.000000,1.000000


## Result

Ran top to bottom without errors — `src/cleaning.py` (06), `src/outliers.py` (07),
`src/eda.py` (08), and `src/features.py` (09) all work against the current project
dataset. Per-stage write-ups: `homework/homework06/`–`homework09/`,
`notebooks/sensitivity_outliers.ipynb`, `notebooks/eda.ipynb`, `docs/`, and the README
feature table.